In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearnex import patch_sklearn

patch_sklearn()
from sklearn.metrics import roc_auc_score

PATH = "../../data/ieee"  # thư mục chứa train_transaction.csv
SEED = 42
TARGET = "isFraud"
START_DATE = pd.Timestamp("2017-11-30")

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [ ]:
df = pd.read_csv(f"{PATH}/train_transaction.csv")

f64 = df.select_dtypes("float64").columns
df[f64] = df[f64].astype("float32")

dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
df["DT_M"] = (dt.dt.year - 2017) * 12 + dt.dt.month

features = [
    c for c in df.columns if c not in ["TransactionID", "TransactionDT", TARGET, "DT_M"]
]

for c in df[features].select_dtypes(exclude="number").columns:
    df[c] = df[c].astype("category")
df = df.copy()
last = df["DT_M"].max()
train = df[df["DT_M"] < last - 1]
test = df[df["DT_M"] == last]
print("train:", train.shape, "| test:", test.shape, "| số feature:", len(features))

C:\Users\Thaibh\AppData\Local\Temp\ipykernel_13796\10194439.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_M"] = (dt.dt.year - 2017) * 12 + dt.dt.month


train: (417559, 395) | test: (89326, 395) | số feature: 391


In [5]:
test[features].info()

<class 'pandas.DataFrame'>
RangeIndex: 89326 entries, 501214 to 590539
Columns: 391 entries, TransactionAmt to V339
dtypes: category(14), float32(376), int64(1)
memory usage: 130.0 MB


In [12]:
model = lgb.LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1)
model.fit(train[features], train[TARGET])

pred = model.predict_proba(test[features])[:, 1]
print("AUC:", roc_auc_score(test[TARGET], pred))


AUC: 0.8947451583184837


In [ ]:
from util import summary

print("AUC:", average_precision_score(test[TARGET], pred))

AUC: 0.518323102610124


In [9]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report


X_train = train[features]
y_train = train[TARGET]
X_test = test[features]
y_test = test[TARGET]

# tách loại cột
cat_features = X_train.select_dtypes(include=["category", "object"]).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"num: {len(num_features)} | cat: {len(cat_features)}")


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ]
)


categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore", sparse_output=True, min_frequency=0.01
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,  # giữ sparse để không OOM
)


clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                penalty="l2",
                C=1.0,
                class_weight="balanced",
                max_iter=500,
                n_jobs=-1,
                verbose=1,
            ),
        ),
    ]
)

clf.fit(X_train, y_train)

# 4. Đánh giá
y_pred_proba = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC test (tháng cuối): {auc:.4f}")

print(classification_report(y_test, clf.predict(X_test)))

num: 377 | cat: 14


d:\HUST\GDG\Nhom01_GDG\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


max_iter reached after 950 seconds


d:\HUST\GDG\Nhom01_GDG\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


AUC test (tháng cuối): 0.8273
              precision    recall  f1-score   support

           0       0.99      0.82      0.89     86212
           1       0.12      0.69      0.20      3114

    accuracy                           0.81     89326
   macro avg       0.55      0.75      0.55     89326
weighted avg       0.96      0.81      0.87     89326



In [ ]:
from util import summary

summary(y_test, y_pred_proba)

accuracy_score 0.812708505922128
recall_score: 0.6868978805394991
precision_score: 0.11953727506426735
f1_score 0.20363670982482865
ROC-AUC score: 0.8273083598255094
PRAUC score: 0.16525714244959425
COnfusion matrix: [[70457 15755]
 [  975  2139]]
